# LLM & RAG Structuration Workflow

![](https://github.com/mistralai/cookbook/blob/main/images/rag.png?raw=1)


Retrieval-augmented generation (RAG) is an AI framework that synergizes the capabilities of LLMs and information retrieval systems.

It’s useful to answer questions or generate content leveraging external knowledge. There are two main steps in RAG:
1. retrieval: retrieve relevant information from a knowledge base with text embeddings stored in a vector store
2. generation: insert the relevant information to the prompt for the LLM to generate information.

In this guide, we will walk through a very basic example of RAG with four implementations:

- RAG from scratch with Mistral
- RAG with Mistral and LangChain
- RAG with Mistral and LlamaIndex
- RAG with Mistral and Haystack

<!-- ## RAG from scratch

This section aims to guide you through the process of building a basic RAG from scratch. We have two goals: firstly, to offer users a comprehensive understanding of the internal workings of RAG and demystify the underlying mechanisms; secondly, to empower you with the essential foundations needed to build an RAG using the minimum required dependencies. -->


### Required libraries

In [35]:
# !pip install mistralai
# !pip install langchain 
# !pip install --upgrade numpy
# !pip install --force-reinstall faiss-cpu
# !pip install --upgrade sentence-transformers

In [62]:
from mistralai import Mistral
from langchain.document_loaders import CSVLoader
import faiss, os, numpy as np

### API Key

In [37]:
API_KEY = input("Enter your API Key: ")
client = Mistral(api_key=API_KEY)

## RAG Setting

### Load data

- CSV to text format

In [38]:
# Load the CSV file
DATASET_CSV_PATH = input("Enter the path to your CSV dataset: ")

# Load documents from the CSV
doc = CSVLoader(file_path=DATASET_CSV_PATH).load()

# Save the documents in text format
with open("../data/fighters.txt", "w") as f:
    for line in doc:
        if "name" in line.page_content.strip():  # Check if the content is not empty
            f.write("\n")
        f.write(line.page_content + "\n")


In [39]:
with open("../data/fighters.txt", "r") as f:
    text = f.read()

text

"\nname: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\n\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\n\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\n\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach_cm: 184.9882\n\nname: muhammad ali\nwins: 56\nlooses: 5\ndraws: 0\nko_rate: 0.607\nstance: Orthodox\nage: 80\ncountry: United States\nheight_cm: 191.1096\nreach_cm: 197.99300000000002\n\nname: juan zurita\nwins: 130\nlooses: 23\ndraws: 1\nko_rate: 0.299\nstance: Orthodox\nage: 105\ncountry: Mexico\nheight_cm: 164.8968\nreach_cm: 167.9956\n\nname: fritz

### Split document into chunks

In a RAG system, it is crucial **to split the document into smaller chunks** so that it’s more effective to identify and retrieve the most relevant information in the retrieval process later.

In [40]:
def set_chunks(text, chunk_size=2048):
    chunks = [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
    ]
    return chunks

In [41]:
CHUNK_SIZE = 2048
chunks = set_chunks(text)

In [42]:
print(chunks)

print("Number of chunks: ", len(chunks))

['\nname: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\n\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\n\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\n\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach_cm: 184.9882\n\nname: muhammad ali\nwins: 56\nlooses: 5\ndraws: 0\nko_rate: 0.607\nstance: Orthodox\nage: 80\ncountry: United States\nheight_cm: 191.1096\nreach_cm: 197.99300000000002\n\nname: juan zurita\nwins: 130\nlooses: 23\ndraws: 1\nko_rate: 0.299\nstance: Orthodox\nage: 105\ncountry: Mexico\nheight_cm: 164.8968\nreach_cm: 167.9956\n\nname: frit

<!-- #### Considerations:
- **Chunk size**: Depending on your specific use case, it may be necessary to customize or experiment with different chunk sizes and chunk overlap to achieve optimal performance in RAG. For example, smaller chunks can be more beneficial in retrieval processes, as larger text chunks often contain filler text that can obscure the semantic representation. As such, using smaller text chunks in the retrieval process can enable the RAG system to identify and extract relevant information more effectively and accurately.  However, it’s worth considering the trade-offs that come with using smaller chunks, such as increasing processing time and computational resources.
- **How to split**: While the simplest method is to split the text by character, there are other options depending on the use case and document structure. For example, to avoid exceeding token limits in API calls, it may be necessary to split the text by tokens. To maintain the cohesiveness of the chunks, it can be useful to split the text by sentences, paragraphs, or HTML headers. If working with code, it’s often recommended to split by meaningful code chunks for example using an Abstract Syntax Tree (AST) parser.


### Create embeddings for each text chunk
For each text chunk, we then need to create text embeddings, which are numeric representations of the text in the vector space. Words with similar meanings are expected to be in closer proximity or have a shorter distance in the vector space.
To create an embedding, use Mistral’s embeddings API endpoint and the embedding model `mistral-embed`. We create a `get_text_embedding` to get the embedding from a single text chunk and then we use list comprehension to get text embeddings for all text chunks. -->


In [43]:
def get_text_embedding(input):
    embeddings_batch_response = client.embeddings.create(
          model="mistral-embed",
          inputs=input
      )
    embeddings_value = embeddings_batch_response.data[0].embedding
    
    return embeddings_value

text_embeddings = np.array([get_text_embedding(chunk) for chunk in chunks])

In [44]:
print("Dimension texts: ", text_embeddings.shape)

text_embeddings

Dimension texts:  (44, 1024)


array([[-0.01372528,  0.02931213,  0.00672531, ...,  0.02262878,
         0.02912903, -0.01855469],
       [-0.01593018,  0.01350403,  0.01467133, ...,  0.02680969,
         0.01924133,  0.00255013],
       [-0.00940704,  0.0305481 ,  0.0269165 , ...,  0.0275116 ,
         0.02792358,  0.00256729],
       ...,
       [-0.02120972,  0.02561951,  0.02502441, ...,  0.02441406,
         0.02140808, -0.00306511],
       [-0.02587891,  0.02687073,  0.02200317, ...,  0.02050781,
         0.03105164, -0.00211525],
       [-0.02799988,  0.0209198 ,  0.03686523, ...,  0.03192139,
         0.02941895,  0.01426697]], shape=(44, 1024))

- Save embeddings text format data

In [45]:
np.save(
    "../data/fighters_embeddings.npy",
    text_embeddings
)

### Load into a vector database
Once we get the text embeddings, a common practice is to store them in a vector database for efficient processing and retrieval. There are several vector database to choose from. In our simple example, we are using an open-source vector database Faiss, which allows for efficient similarity search.  

With Faiss, we instantiate an instance of the Index class, which defines the indexing structure of the vector database. We then add the text embeddings to this indexing structure.


In [ ]:
text_embeddings = np.load("../data/fighters_embeddings.npy")

In [47]:
d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

In [ ]:
faiss.write_index(index, "../data/faiss_vectorized_db")

#### Considerations:
- **Vector database**: When selecting a vector database, there are several factors to consider including speed, scalability, cloud management, advanced filtering, and open-source vs. closed-source.

### Create embeddings for a question
Whenever users ask a question, we also need to create embeddings for this question using the same embedding models as before.


In [48]:
def set_outcome():
    return ""

def get_fight_outcome(fighters: list[str], outcome: str, discipline: str = "boxing") -> str:
    return f"""Based on the predicted outcome of {discipline} fight between the fighters {fighters} is: {outcome}"""

def get_prompt(context: str, retrieved_data: str, task: str, constraints: list[str]):
    return f"""
    Context information & required data is below.
    ---------------------
    Context: {context}
    Retrieved data: {retrieved_data}
    ---------------------
    Given the context information and not prior knowledge, answer here's your task:
    {task}
    
    Constraints: {constraints}
    """


In [49]:
FIGHTERS = [
    "Saul Alvarez",
    "Crawford"
]

CONTEXT = f"""
            Be a expert in boxing since its creation in 1880.
            You've access to the following prediction:
            {get_fight_outcome(FIGHTERS, set_outcome())}
           """

TASK = f"""
        Describe in the most detailed way the outcome of this fight between these boxers: {FIGHTERS}.
        Be sure to clearly choose one winner and explain exactly how and why they won, including the method of victory, key moments, and what led to the final result.
        """

CONSTRAINTS = [
    "Use a simple language.",  "Be concise.",  "Have a sport host tone."
]


In [50]:
task_embeddings = np.array(
    [get_text_embedding(TASK)]
)

print("Task embeddings: ", task_embeddings)
print("Task embeddings shape: ", task_embeddings.shape)

Task embeddings:  [[ 0.00224113  0.06234741  0.03771973 ... -0.01652527  0.00068188
  -0.02478027]]
Task embeddings shape:  (1, 1024)


#### Considerations:
- Hypothetical Document Embeddings (HyDE): In some cases, the user’s question might not be the most relevant query to use for identifying the relevant context. Instead, it maybe more effective to generate a hypothetical answer or a hypothetical document based on the user’s query and use the embeddings of the generated text to retrieve similar text chunks.

### Retrieve similar chunks from the vector database
We can perform a search on the vector database with `index.search`, which takes two arguments: the first is the vector of the question embeddings, and the second is the number of similar vectors to retrieve. This function returns the distances and the indices of the most similar vectors to the question vector in the vector database. Then based on the returned indices, we can retrieve the actual relevant text chunks that correspond to those indices.


### Load vector database

In [ ]:
index = faiss.read_index("../data/faiss_vectorized_db")

RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\faiss\faiss\impl\io.cpp:68: Error: 'f' failed: could not open ../data/faiss_vectorized_db for reading: Permission denied

-  **distance** is the distance between the request and the indexed embeddings
-  **indices_of_nearest_n** is the indices of the nearest neighbors

In [ ]:
distance, indices_of_nearest_n = index.search(task_embeddings, k=2)


In [ ]:
print("Indices of nearest neighbors of the embedding content: ", indices_of_nearest_n)
print("Distances between the request and the indexed embeddings: ", distance)

Indices of nearest neighbors of the embedding content:  [[ 0 34]]
Distances between the request and the indexed embeddings:  [[0.29177874 0.31331658]]


In [ ]:
RETRIEVED_DATA = [
    chunks[i] for i in indices_of_nearest_n.tolist()[0]
]

print(RETRIEVED_DATA)

['\nname: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\n\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\n\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\n\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach_cm: 184.9882\n\nname: muhammad ali\nwins: 56\nlooses: 5\ndraws: 0\nko_rate: 0.607\nstance: Orthodox\nage: 80\ncountry: United States\nheight_cm: 191.1096\nreach_cm: 197.99300000000002\n\nname: juan zurita\nwins: 130\nlooses: 23\ndraws: 1\nko_rate: 0.299\nstance: Orthodox\nage: 105\ncountry: Mexico\nheight_cm: 164.8968\nreach_cm: 167.9956\n\nname: frit

In [54]:
request_by_prompt = get_prompt(CONTEXT, RETRIEVED_DATA, TASK, CONSTRAINTS)
print(request_by_prompt)


    Context information & required data is below.
    ---------------------
    Context: 
            Be a expert in boxing since its creation in 1880.
            You've access to the following prediction:
            Based on the predicted outcome of boxing fight between the fighters ['Saul Alvarez', 'Crawford'] is: 
           
    Retrieved data: ['\nname: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\n\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\n\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\n\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach

#### Considerations:
- **Retrieval methods**: There are a lot different retrieval strategies. In our example, we are showing a simple similarity search with embeddings. Sometimes when there is metadata available for the data, it’s better to filter the data based on the metadata first before performing similarity search. There are also other statistical retrieval methods like TF-IDF and BM25 that use frequency and distribution of terms in the document to identify relevant text chunks.
- **Retrieved document**: Do we always retrieve individual text chunk as it is? Not always.
    - Sometimes, we would like to include more context around the actual retrieved text chunk. We call the actual retrieve text chunk “child chunk” and our goal is to retrieve a larger “parent chunk” that the “child chunk” belongs to.
    - On occasion, we might also want to provide weights to our retrieve documents. For example, a time-weighted approach would help us retrieve the most recent document.
    - One common issue in the retrieval process is the “lost in the middle” problem where the information in the middle of a long context gets lost. Our models have tried to mitigate this issue. For example, in the passkey task, our models have demonstrated the ability to find a "needle in a haystack" by retrieving a randomly inserted passkey within a long prompt, up to 32k context length. However, it is worth considering experimenting with reordering the document to determine if placing the most relevant chunks at the beginning and end leads to improved results.
  
### Combine context and question in a prompt and generate response

Finally, we can offer the retrieved text chunks as the context information within the prompt. Here is a prompt template where we can include both the retrieved text and user question in the prompt.



In [55]:
request_by_prompt = get_prompt(CONTEXT, RETRIEVED_DATA, TASK, CONSTRAINTS)
print(request_by_prompt)


    Context information & required data is below.
    ---------------------
    Context: 
            Be a expert in boxing since its creation in 1880.
            You've access to the following prediction:
            Based on the predicted outcome of boxing fight between the fighters ['Saul Alvarez', 'Crawford'] is: 
           
    Retrieved data: ['\nname: saul alvarez\nwins: 54\nlooses: 1\ndraws: 2\nko_rate: 0.632\nstance: Orthodox\nage: 32\ncountry: Mexico\nheight_cm: 174.95520000000002\nreach_cm: 178.9938\n\nname: rene alvarado\nwins: 32\nlooses: 8\ndraws: 0\nko_rate: 0.525\nstance: Orthodox\nage: 33\ncountry: Nicaragua\nheight_cm: 170.07840000000002\nreach_cm: 183.007\n\nname: daniel alicea\nwins: 30\nlooses: 7\ndraws: 2\nko_rate: 0.564\nstance: Orthodox\nage: 49\ncountry: Puerto Rico\nheight_cm: 173.1264\nreach_cm: 178.0032\n\nname: sadam ali\nwins: 27\nlooses: 3\ndraws: 0\nko_rate: 0.467\nstance: Orthodox\nage: 33\ncountry: United States\nheight_cm: 174.95520000000002\nreach

In [ ]:
class MistralLLM:
    
    def __init__(self, client, model):
        self.client = client
        self.model = model
    
    def run(self, request):
        messages = [
            {
                "role": "user", "content": request
            }
        ]
        chat_response = self.client.chat.complete(
            model=self.model,
            messages=messages
        )
        return (chat_response.choices[0].message.content)

In [ ]:
global MODELS
MODELS = {
    "default": "open-mistral-7b",          # Correct name for 7B model
    "fast": "mistral-tiny",               # Fastest model
    # "medium": "mistral-small",            # Medium model
    # "large": "mistral-medium",            # Larger model
    "latest": "mistral-large-latest"      # Latest large model
}

In [ ]:
llm = MistralLLM(client, MODELS["fast"])

In [ ]:
# output = LLM.run(request_by_prompt, model=MODELS["fast"])
output = llm.run(request_by_prompt)

'In the highly anticipated bout between Saul "Canelo" Alvarez and Terence Crawford, Canelo emerged as the victor in a hard-fought and thrilling 12-round decision.\n\nCanelo, the experienced and formidable Mexican champion, displayed his exceptional boxing skills throughout the fight, consistently landing accurate jabs and powerful straight right hands. His orthodox stance, coupled with his quick footwork and agility, allowed him to maneuver around the ring effectively, making it difficult for Crawford to land his punches.\n\nCrawford, a skilled and aggressive fighter, pressed forward with his combination punches, but often found himself missing his mark due to Canelo\'s elusive movements. While Crawford managed to land some solid shots, he was unable to consistently out-box Canelo, who maintained a steady pace and controlled the tempo of the fight.\n\nIn the final rounds, Canelo began to assert his dominance, landing a series of punishing combinations that left Crawford visibly wobbled